# Morpheus Inpatient Workbench

Welcome to the **Morpheus** inpatient analytics workspace — Parthenon's ICU and hospital outcomes research module.

This notebook provides direct SQL access to MIMIC-IV (and other inpatient datasets) loaded into the Parthenon database, plus helper functions for common ICU research patterns:

1. **Connect** — database & Morpheus API setup
2. **Explore datasets** — discover available inpatient schemas
3. **Population overview** — admissions, demographics, mortality
4. **ICU analytics** — unit stays, ventilation, sedation, vitals
5. **Patient deep-dive** — single-patient journey reconstruction
6. **Temporal trends** — admission volumes, LOS, mortality over time
7. **Publication-ready charts** — dark clinical theme matching Parthenon UI

---

**Named after the god of dreams** — appropriate for ICU care where sedation, delirium monitoring, and the ABCDEF Liberation Bundle are central concerns.

## 1. Environment & Connection Setup

Run this cell first — it establishes your database connection, API client, and Morpheus-specific helpers.

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import numpy as np
import requests
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

# ── Paths ──
NOTEBOOK_DIR = Path(os.environ.get('PARTHENON_NOTEBOOK_DIR', '/home/jovyan/notebooks'))
SHARED_DIR = Path('/home/jovyan/shared')
USER_EMAIL = os.environ.get('PARTHENON_USER_EMAIL', '')

# ── Database ──
DB_USER = os.environ.get('PARTHENON_DB_USER', 'parthenon')
DB_PASS = os.environ.get('PARTHENON_DB_PASSWORD', '')
DB_HOST = os.environ.get('PARTHENON_DB_HOST', 'postgres')
DB_PORT = os.environ.get('PARTHENON_DB_PORT', '5432')
DB_NAME = os.environ.get('PARTHENON_DB_NAME', 'parthenon')

DATABASE_URL = f'postgresql+psycopg://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(DATABASE_URL)

# ── API client ──
API_BASE = os.environ.get('PARTHENON_API_BASE_URL', 'http://nginx/api/v1')
API_TOKEN = os.environ.get('PARTHENON_API_TOKEN', '')

def api_get(path: str, params: dict | None = None) -> dict:
    """GET request to Parthenon API."""
    headers = {'Accept': 'application/json'}
    if API_TOKEN:
        headers['Authorization'] = f'Bearer {API_TOKEN}'
    url = urljoin(API_BASE.rstrip('/') + '/', path.lstrip('/'))
    resp = requests.get(url, params=params, headers=headers, timeout=30)
    resp.raise_for_status()
    return resp.json()

def query(sql: str, params: dict | None = None) -> pd.DataFrame:
    """Run a SQL query and return a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

# ── Default schema (MIMIC-IV) ──
SCHEMA = os.environ.get('MORPHEUS_SCHEMA', 'mimiciv')

# ── Verify ──
print(f'\u2713 User: {USER_EMAIL}')
print(f'\u2713 DB role: {DB_USER}')
print(f'\u2713 Default schema: {SCHEMA}')

with engine.connect() as conn:
    result = conn.execute(text('SELECT current_user, current_database()'))
    row = result.fetchone()
    print(f'\u2713 Connected to {row[1]} as {row[0]}')

## 2. Discover Available Datasets

Morpheus supports multiple inpatient datasets (MIMIC-IV, real EHR integrations). Discover what's loaded.

In [ ]:
# Query the dataset registry
datasets = query("""
    SELECT dataset_id, name, schema_name, description, source_type, patient_count, status
    FROM app.morpheus_datasets
    WHERE status = 'active'
    ORDER BY name
""")

if datasets.empty:
    print('No datasets in registry — falling back to mimiciv schema')
    print('Available schemas:')
    schemas = query("SELECT schema_name FROM information_schema.schemata WHERE schema_name LIKE '%mimic%' OR schema_name LIKE '%inpatient%' ORDER BY 1")
    display(schemas)
else:
    display(datasets)

In [ ]:
def set_schema(schema_name: str) -> None:
    """Switch the active Morpheus schema for subsequent queries."""
    global SCHEMA
    # Validate: alphanumeric + underscore only
    import re
    if not re.match(r'^[a-zA-Z0-9_]+$', schema_name):
        raise ValueError(f'Invalid schema name: {schema_name}')
    SCHEMA = schema_name
    print(f'\u2713 Active schema: {SCHEMA}')

# Example: switch to a different dataset
# set_schema('mimiciv')

## 3. Population Overview

Key metrics for the selected inpatient dataset — mirrors the Morpheus dashboard in the Parthenon web UI.

In [ ]:
metrics = query(f"""
    SELECT
        COUNT(DISTINCT p.subject_id) AS total_patients,
        COUNT(DISTINCT a.hadm_id) AS total_admissions,
        ROUND(100.0 * COUNT(DISTINCT i.stay_id) / NULLIF(COUNT(DISTINCT a.hadm_id), 0), 1) AS icu_rate_pct,
        ROUND(100.0 * SUM(CASE WHEN a.hospital_expire_flag = 1 THEN 1 ELSE 0 END)::numeric
              / NULLIF(COUNT(DISTINCT a.hadm_id), 0), 1) AS mortality_rate_pct,
        ROUND(AVG(EXTRACT(EPOCH FROM (a.dischtime - a.admittime)) / 86400)::numeric, 1) AS avg_los_days
    FROM {SCHEMA}.patients p
    LEFT JOIN {SCHEMA}.admissions a ON p.subject_id = a.subject_id
    LEFT JOIN {SCHEMA}.icustays i ON a.hadm_id = i.hadm_id
""")

print('=== Population Metrics ===')
for col in metrics.columns:
    print(f'  {col}: {metrics[col].iloc[0]}')

In [ ]:
# Gender distribution
gender = query(f"""
    SELECT gender, COUNT(*) AS n
    FROM {SCHEMA}.patients
    GROUP BY gender
    ORDER BY n DESC
""")
display(gender)

In [ ]:
# Age distribution at admission
ages = query(f"""
    SELECT
        CASE
            WHEN anchor_age < 30 THEN '18-29'
            WHEN anchor_age < 40 THEN '30-39'
            WHEN anchor_age < 50 THEN '40-49'
            WHEN anchor_age < 60 THEN '50-59'
            WHEN anchor_age < 70 THEN '60-69'
            WHEN anchor_age < 80 THEN '70-79'
            ELSE '80+'
        END AS age_group,
        COUNT(*) AS n
    FROM {SCHEMA}.patients
    GROUP BY 1
    ORDER BY MIN(anchor_age)
""")
display(ages)

## 4. ICU Analytics

Deep-dive into ICU admissions: unit types, length of stay, ventilation patterns.

In [ ]:
# ICU unit statistics
icu_units = query(f"""
    SELECT
        first_careunit AS unit,
        COUNT(*) AS stays,
        ROUND(AVG(los)::numeric, 1) AS avg_los_days,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY los)::numeric, 1) AS median_los_days,
        MAX(los) AS max_los_days
    FROM {SCHEMA}.icustays
    GROUP BY first_careunit
    ORDER BY stays DESC
""")
display(icu_units)

In [ ]:
# Mortality by admission type
mortality = query(f"""
    SELECT
        admission_type,
        COUNT(*) AS total,
        SUM(hospital_expire_flag) AS deaths,
        ROUND(100.0 * SUM(hospital_expire_flag) / COUNT(*)::numeric, 1) AS mortality_pct
    FROM {SCHEMA}.admissions
    GROUP BY admission_type
    ORDER BY total DESC
""")
display(mortality)

In [ ]:
# Top 20 diagnoses across all admissions
top_dx = query(f"""
    SELECT d.icd_code, d.icd_version,
           COALESCE(di.long_title, d.icd_code) AS description,
           COUNT(DISTINCT d.subject_id) AS patients,
           COUNT(DISTINCT d.hadm_id) AS admissions
    FROM {SCHEMA}.diagnoses_icd d
    LEFT JOIN {SCHEMA}.d_icd_diagnoses di
        ON d.icd_code = di.icd_code AND d.icd_version = di.icd_version
    WHERE d.seq_num = 1
    GROUP BY d.icd_code, d.icd_version, di.long_title
    ORDER BY patients DESC
    LIMIT 20
""")
display(top_dx)

## 5. Patient Deep-Dive

Reconstruct a single patient's hospital journey — admissions, transfers, diagnoses, medications, labs, vitals.

In [ ]:
def patient_admissions(subject_id: int) -> pd.DataFrame:
    """All admissions for a patient."""
    return query(f"""
        SELECT hadm_id, admittime, dischtime, deathtime,
               admission_type, admission_location, discharge_location,
               insurance, marital_status, race, hospital_expire_flag,
               ROUND(EXTRACT(EPOCH FROM (dischtime - admittime)) / 86400, 1) AS los_days
        FROM {SCHEMA}.admissions
        WHERE subject_id = :sid
        ORDER BY admittime
    """, {'sid': subject_id})

def patient_transfers(subject_id: int, hadm_id: int | None = None) -> pd.DataFrame:
    """Transfer events (care unit movements) for a patient."""
    sql = f"""
        SELECT transfer_id, hadm_id, eventtype, careunit, intime, outtime,
               ROUND(EXTRACT(EPOCH FROM (outtime - intime)) / 3600, 1) AS duration_hours
        FROM {SCHEMA}.transfers
        WHERE subject_id = :sid
    """
    params = {'sid': subject_id}
    if hadm_id:
        sql += ' AND hadm_id = :hadm'
        params['hadm'] = hadm_id
    sql += ' ORDER BY intime'
    return query(sql, params)

def patient_diagnoses(subject_id: int, hadm_id: int | None = None) -> pd.DataFrame:
    """Diagnoses for a patient, optionally filtered by admission."""
    sql = f"""
        SELECT d.hadm_id, d.seq_num, d.icd_code, d.icd_version,
               COALESCE(di.long_title, d.icd_code) AS description
        FROM {SCHEMA}.diagnoses_icd d
        LEFT JOIN {SCHEMA}.d_icd_diagnoses di
            ON d.icd_code = di.icd_code AND d.icd_version = di.icd_version
        WHERE d.subject_id = :sid
    """
    params = {'sid': subject_id}
    if hadm_id:
        sql += ' AND d.hadm_id = :hadm'
        params['hadm'] = hadm_id
    sql += ' ORDER BY d.hadm_id, d.seq_num'
    return query(sql, params)

def patient_medications(subject_id: int, hadm_id: int | None = None) -> pd.DataFrame:
    """Medication orders for a patient."""
    sql = f"""
        SELECT hadm_id, starttime, stoptime, drug, drug_type,
               route, dose_val_rx, dose_unit_rx
        FROM {SCHEMA}.prescriptions
        WHERE subject_id = :sid
    """
    params = {'sid': subject_id}
    if hadm_id:
        sql += ' AND hadm_id = :hadm'
        params['hadm'] = hadm_id
    sql += ' ORDER BY starttime'
    return query(sql, params)

def patient_labs(subject_id: int, hadm_id: int | None = None, limit: int = 500) -> pd.DataFrame:
    """Lab results for a patient."""
    sql = f"""
        SELECT le.hadm_id, le.charttime, di.label, di.fluid, di.category,
               le.value, le.valuenum, le.valueuom, le.ref_range_lower, le.ref_range_upper, le.flag
        FROM {SCHEMA}.labevents le
        JOIN {SCHEMA}.d_labitems di ON le.itemid = di.itemid
        WHERE le.subject_id = :sid
    """
    params = {'sid': subject_id}
    if hadm_id:
        sql += ' AND le.hadm_id = :hadm'
        params['hadm'] = hadm_id
    sql += f' ORDER BY le.charttime LIMIT {int(limit)}'
    return query(sql, params)

def patient_vitals(subject_id: int, stay_id: int | None = None, limit: int = 1000) -> pd.DataFrame:
    """Charted vitals/observations for a patient's ICU stay."""
    sql = f"""
        SELECT ce.stay_id, ce.charttime, di.label, di.abbreviation, di.category,
               ce.value, ce.valuenum, ce.valueuom
        FROM {SCHEMA}.chartevents ce
        JOIN {SCHEMA}.d_items di ON ce.itemid = di.itemid
        WHERE ce.subject_id = :sid
    """
    params = {'sid': subject_id}
    if stay_id:
        sql += ' AND ce.stay_id = :stay'
        params['stay'] = stay_id
    sql += f' ORDER BY ce.charttime LIMIT {int(limit)}'
    return query(sql, params)

print('\u2713 Patient helper functions ready')

In [ ]:
# Pick a sample patient
sample = query(f"""
    SELECT p.subject_id, p.gender, p.anchor_age,
           COUNT(DISTINCT a.hadm_id) AS admissions,
           COUNT(DISTINCT i.stay_id) AS icu_stays
    FROM {SCHEMA}.patients p
    JOIN {SCHEMA}.admissions a ON p.subject_id = a.subject_id
    LEFT JOIN {SCHEMA}.icustays i ON a.hadm_id = i.hadm_id
    GROUP BY p.subject_id, p.gender, p.anchor_age
    HAVING COUNT(DISTINCT i.stay_id) >= 1
    ORDER BY COUNT(DISTINCT a.hadm_id) DESC
    LIMIT 5
""")
display(sample)

# Uncomment to explore a specific patient:
# PATIENT_ID = sample['subject_id'].iloc[0]
# display(patient_admissions(PATIENT_ID))
# display(patient_diagnoses(PATIENT_ID))
# display(patient_medications(PATIENT_ID))

## 6. Temporal Trends

Monthly admission volumes, mortality rates, and length of stay over time.

In [ ]:
trends = query(f"""
    SELECT
        DATE_TRUNC('month', admittime) AS month,
        COUNT(*) AS admissions,
        SUM(hospital_expire_flag) AS deaths,
        ROUND(100.0 * SUM(hospital_expire_flag) / COUNT(*)::numeric, 1) AS mortality_pct,
        ROUND(AVG(EXTRACT(EPOCH FROM (dischtime - admittime)) / 86400)::numeric, 1) AS avg_los
    FROM {SCHEMA}.admissions
    WHERE admittime IS NOT NULL AND dischtime IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""")
print(f'{len(trends)} months of data')
trends.head(10)

## 7. Visualizations

Publication-quality dark-themed charts matching the Parthenon clinical UI.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')

COLORS = {
    'bg': '#0E0E11',
    'panel': '#18181B',
    'border': '#27272A',
    'text': '#C5C0B8',
    'crimson': '#9B1B30',
    'gold': '#C9A227',
    'teal': '#2DD4BF',
    'muted': '#71717A',
}

def morpheus_fig(width=12, height=6, title=''):
    """Create a Parthenon-themed figure."""
    fig, ax = plt.subplots(figsize=(width, height), facecolor=COLORS['bg'])
    ax.set_facecolor(COLORS['bg'])
    ax.tick_params(colors=COLORS['text'], labelsize=9)
    for spine in ax.spines.values():
        spine.set_color(COLORS['border'])
    if title:
        ax.set_title(title, color=COLORS['text'], fontsize=14, fontweight='bold', pad=12)
    return fig, ax

print('\u2713 Visualization helpers ready')

In [ ]:
# Monthly admissions trend
if not trends.empty:
    fig, ax = morpheus_fig(14, 5, 'Monthly Admissions')
    ax.fill_between(trends['month'], trends['admissions'], alpha=0.2, color=COLORS['teal'])
    ax.plot(trends['month'], trends['admissions'], color=COLORS['teal'], linewidth=1.5)
    ax.set_xlabel('Month', color=COLORS['muted'])
    ax.set_ylabel('Admissions', color=COLORS['muted'])
    plt.tight_layout()
    plt.show()

In [ ]:
# Mortality by admission type
if not mortality.empty:
    fig, ax = morpheus_fig(10, 5, 'Mortality Rate by Admission Type')
    bars = ax.barh(mortality['admission_type'], mortality['mortality_pct'],
                   color=COLORS['crimson'], edgecolor=COLORS['border'], height=0.6)
    ax.set_xlabel('Mortality %', color=COLORS['muted'])
    for bar, pct in zip(bars, mortality['mortality_pct']):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{pct}%', va='center', color=COLORS['text'], fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# ICU LOS distribution
los_dist = query(f"""
    SELECT
        CASE
            WHEN los < 1 THEN '< 1 day'
            WHEN los < 3 THEN '1-2 days'
            WHEN los < 7 THEN '3-6 days'
            WHEN los < 14 THEN '7-13 days'
            ELSE '14+ days'
        END AS bucket,
        COUNT(*) AS stays
    FROM {SCHEMA}.icustays
    GROUP BY 1
    ORDER BY MIN(los)
""")

if not los_dist.empty:
    fig, ax = morpheus_fig(8, 5, 'ICU Length of Stay Distribution')
    ax.bar(los_dist['bucket'], los_dist['stays'],
           color=COLORS['gold'], edgecolor=COLORS['border'])
    ax.set_xlabel('LOS Bucket', color=COLORS['muted'])
    ax.set_ylabel('ICU Stays', color=COLORS['muted'])
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

## 8. Quick Recipe Reference

Copy these into new cells for common inpatient research tasks.

In [ ]:
recipes = pd.DataFrame([
    {'task': 'List datasets', 'code': "query('SELECT * FROM app.morpheus_datasets')"},
    {'task': 'Switch schema', 'code': "set_schema('mimiciv')"},
    {'task': 'Population metrics', 'code': f"query('SELECT COUNT(DISTINCT subject_id) FROM {SCHEMA}.patients')"},
    {'task': 'Patient admissions', 'code': 'patient_admissions(10006)'},
    {'task': 'Patient diagnoses', 'code': 'patient_diagnoses(10006, hadm_id=21234567)'},
    {'task': 'Patient medications', 'code': 'patient_medications(10006)'},
    {'task': 'Patient labs', 'code': 'patient_labs(10006, limit=100)'},
    {'task': 'Patient vitals', 'code': 'patient_vitals(10006, stay_id=30012345)'},
    {'task': 'Patient transfers', 'code': 'patient_transfers(10006)'},
    {'task': 'Top diagnoses', 'code': f"query('SELECT ... FROM {SCHEMA}.diagnoses_icd ... LIMIT 20')"},
    {'task': 'ICU unit stats', 'code': f"query('SELECT first_careunit, COUNT(*) FROM {SCHEMA}.icustays GROUP BY 1')"},
    {'task': 'Monthly trends', 'code': f"query('SELECT DATE_TRUNC(month, admittime), COUNT(*) FROM {SCHEMA}.admissions GROUP BY 1')"},
    {'task': 'Morpheus API', 'code': "api_get('/morpheus/dashboard/metrics?dataset=mimiciv')"},
    {'task': 'Export results', 'code': "df.to_csv(NOTEBOOK_DIR / 'output' / 'my_results.csv', index=False)"},
])
recipes

---

## Next Steps

1. **Explore the dataset** — run the population overview cells to understand your data
2. **Pick a patient** — use the sample query to find interesting cases, then deep-dive
3. **Build cohorts** — filter by diagnosis, ICU status, mortality to define study populations
4. **Visualize** — use `morpheus_fig()` for publication-ready dark-themed charts
5. **Export** — save DataFrames to CSV/Parquet for use in R, HADES, or external tools

**Morpheus schemas:** `mimiciv.patients`, `mimiciv.admissions`, `mimiciv.icustays`, `mimiciv.transfers`, `mimiciv.diagnoses_icd`, `mimiciv.prescriptions`, `mimiciv.labevents`, `mimiciv.chartevents`, `mimiciv.microbiologyevents`

**Parthenon API:** Use `api_get('/morpheus/dashboard/metrics?dataset=mimiciv')` to access the same data the web UI shows.

Your server stops after 30 minutes of inactivity. All work is saved automatically.